## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import copy
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import Ridge, Lasso
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr

# Load clean transaction data
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date", "pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date", "pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:", df_train.shape)
print("Valid transactions:", df_valid.shape)
print("Train targets:     ", df_train_targets.shape)
print("Valid targets:     ", df_valid_targets.shape)

# Reference date: last day of the transaction period
# Used to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")

Train transactions: (219128, 27)
Valid transactions: (55183, 27)
Train targets:      (93272, 2)
Valid targets:      (23319, 2)


## 2. Feature Engineering
All features are aggregated at customer level (one row per customer).  
We build three feature blocks and merge them:
- **RFM + behavior features** — recency, frequency, monetary, returns, discounts, delivery, trend
- **Seasonality features** — when during the year the customer buys
- **Product profile features** — what types of products the customer buys

In [2]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input:  transaction-level dataframe (one row per item)
    Output: one row per customer
    """
    df = df.copy()
    # Avoid categorical dtype issues that can cause memory errors
    df["cust_id"] = df["cust_id"].astype(str)
    df["sale_id"]  = df["sale_id"].astype(str)

    # Separate returned and non-returned items
    df_net = df[df["is_returned"] == 0]

    # ── Recency: days since last purchase ──────────────────────────────────
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date - x).days)
        .rename("recency_days")
    )

    # ── Frequency: unique orders with at least 1 non-returned item ─────────
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )

    # ── One-time buyer flag ─────────────────────────────────────────────────
    is_one_time_buyer = (frequency == 1).astype(int).rename("is_one_time_buyer")

    # ── Days between orders (only meaningful for repeat buyers) ────────────
    order_dates_per_cust = (
        df.drop_duplicates(subset=["cust_id", "sale_id"])
        .sort_values(["cust_id", "order_date"])
        .groupby("cust_id")["order_date"]
    )
    avg_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.mean() if len(x) > 1 else np.nan)
        .rename("avg_days_between_orders")
    )
    std_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.std() if len(x) > 1 else np.nan)
        .rename("std_days_between_orders")
    )

    # ── Monetary: total revenue excluding returns ───────────────────────────
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )

    # ── Average ticket: revenue per effective order ─────────────────────────
    avg_ticket = (total_revenue / frequency).rename("avg_ticket")

    # ── Items per order (gross): includes returns ───────────────────────────
    items_per_order_gross = (
        df.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_gross")
    )

    # ── Items per order (net): excludes returns ─────────────────────────────
    items_per_order_net = (
        df_net.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    # ── Return behavior ─────────────────────────────────────────────────────
    total_returns = df.groupby("cust_id")["is_returned"].sum().rename("total_returns")
    total_items   = df.groupby("cust_id")["is_returned"].count().rename("total_items")
    return_rate   = (total_returns / total_items).rename("return_rate")

    # ── Delivery time ───────────────────────────────────────────────────────
    df["delivery_days"] = (df["pack_date"] - df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    # ── Discount behavior ───────────────────────────────────────────────────
    df["has_discount_item"] = (df["sale_discount_applied"] < 0).astype(int)
    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"].sum().rename("total_discounted_items")
    )
    discount_rate = (
        df.groupby("cust_id")["has_discount_item"].mean().rename("discount_rate")
    )

    # ── Trend features: is the customer becoming more active over time? ─────
    df["year"] = df["order_date"].dt.year
    orders_2017  = df[df["year"] == 2017].groupby("cust_id").size()
    orders_total = df.groupby("cust_id").size()
    pct_orders_2017 = (orders_2017 / orders_total).fillna(0).rename("pct_orders_2017")

    revenue_2016 = (
        df[df["year"] == 2016].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2016")
    )
    revenue_2017 = (
        df[df["year"] == 2017].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2017")
    )
    revenue_trend = (revenue_2017 - revenue_2016).rename("revenue_trend")

    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket,
        items_per_order_gross, items_per_order_net,
        total_returns, total_items, return_rate,
        total_discounted_items, discount_rate,
        avg_days_between_orders, std_days_between_orders,
        is_one_time_buyer, pct_orders_2017, revenue_trend
    ], axis=1).reset_index()

    return rfm

In [3]:
def compute_seasonality_features(df):
    """
    Captures when during the year the customer tends to buy.
    We focus on January and July (peak sales months).
    most_active_month was discarded due to too many ties.
    """
    df = df.copy()
    df["order_month"] = df["order_date"].dt.month

    buys_in_jan = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 1).any()))
        .rename("buys_in_jan")
    )
    buys_in_jul = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 7).any()))
        .rename("buys_in_jul")
    )
    return pd.concat([buys_in_jan, buys_in_jul], axis=1).reset_index()

In [4]:
def compute_product_features(df):
    """
    Captures diversity of purchasing behavior across product categories,
    brands, and customer segments.
    """
    df = df.copy()

    n_genders = df.groupby("cust_id")["prod_type_1"].nunique().rename("n_genders")
    buys_women = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "women").any())).rename("buys_women")
    )
    buys_men = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "men").any())).rename("buys_men")
    )
    buys_kids = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int(x.isin(["boys", "girls"]).any())).rename("buys_kids")
    )
    n_product_categories = (
        df.groupby("cust_id")["prod_type_3"].nunique().rename("n_product_categories")
    )
    n_brands = (
        df.groupby("cust_id")["prod_brand"].nunique().rename("n_brands")
    )
    pct_web_only = (
        df.groupby("cust_id")["prod_web_only"].mean().rename("pct_web_only")
    )
    buys_outlet = (
        df.groupby("cust_id")["prod_outlet"]
        .apply(lambda x: int((x > 0).any())).rename("buys_outlet")
    )
    return pd.concat([
        n_genders, buys_women, buys_men, buys_kids,
        n_product_categories, n_brands, pct_web_only, buys_outlet
    ], axis=1).reset_index()

In [5]:
def build_features(df):
    """Combines all feature blocks into one customer-level table."""
    rfm         = compute_rfm_features(df)
    seasonality = compute_seasonality_features(df)
    product     = compute_product_features(df)
    return rfm.merge(seasonality, on="cust_id").merge(product, on="cust_id")

df_train_features = build_features(df_train)
df_valid_features = build_features(df_valid)

print("Train features shape:", df_train_features.shape)
print("Valid features shape:", df_valid_features.shape)

Train features shape: (93272, 27)
Valid features shape: (23319, 27)


In [6]:
# Merge features with targets
df_train_final = df_train_features.merge(df_train_targets, on="cust_id")
df_valid_final = df_valid_features.merge(df_valid_targets, on="cust_id")

print("Train final shape:", df_train_final.shape)
print("Valid final shape:", df_valid_final.shape)

Train final shape: (93272, 28)
Valid final shape: (23319, 28)


## 4. NA Cleaning

In [7]:
target = df_train_final["revenue_2018_2019"]
print("==== Target distribution ====")
print(f"Customers with revenue = 0: {(target==0).sum()} ({100*(target==0).mean():.1f}%)")
print(f"Customers with revenue > 0: {(target>0).sum()} ({100*(target>0).mean():.1f}%)")
print(f"\nMean (all):     {target.mean():.2f}")
print(f"Mean (>0 only): {target[target>0].mean():.2f}")
print(f"Max:            {target.max():.2f}")

print("\n=== NaNs per column ===")
nans = df_train_final.isna().sum()
print(nans[nans > 0])

==== Target distribution ====
Customers with revenue = 0: 59196 (63.5%)
Customers with revenue > 0: 34076 (36.5%)

Mean (all):     70.18
Mean (>0 only): 192.08
Max:            1197.94

=== NaNs per column ===
frequency                      7
total_revenue                  7
avg_ticket                     7
items_per_order_net            7
avg_days_between_orders    61971
std_days_between_orders    78984
is_one_time_buyer              7
revenue_trend              77146
dtype: int64


In [8]:
# ── NaN handling ─────────────────────────────────────────────────────────────
# Customers who returned everything → fill with 0
cols_fill_zero = ["frequency", "total_revenue", "avg_ticket",
                  "items_per_order_net", "is_one_time_buyer"]
df_train_final[cols_fill_zero] = df_train_final[cols_fill_zero].fillna(0)
df_valid_final[cols_fill_zero] = df_valid_final[cols_fill_zero].fillna(0)

# revenue_trend NaN = customer only bought in one year → no trend → fill with 0
df_train_final["revenue_trend"] = df_train_final["revenue_trend"].fillna(0)
df_valid_final["revenue_trend"] = df_valid_final["revenue_trend"].fillna(0)

# One-time buyers → fill days between orders with train median
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    median_val = df_train_final[col].median()
    df_train_final[col] = df_train_final[col].fillna(median_val)
    df_valid_final[col] = df_valid_final[col].fillna(median_val)

print("Remaining NaNs in train:", df_train_final.isna().sum().sum())
print("Remaining NaNs in valid:", df_valid_final.isna().sum().sum())

Remaining NaNs in train: 0
Remaining NaNs in valid: 0


In [9]:
# ── VIP feature ──────────────────────────────────────────────────────────────
# Identified through error analysis: high-value customers are systematically underpredicted
# Thresholds based on train distribution: frequency >= 4 (top quartile), revenue >= 249 (top 10%)
# NOTE: thresholds derived from train only → no data leaking
df_train_final["is_vip"] = (
    (df_train_final["frequency"] >= 4) &
    (df_train_final["total_revenue"] >= 249)
).astype(int)
df_valid_final["is_vip"] = (
    (df_valid_final["frequency"] >= 4) &
    (df_valid_final["total_revenue"] >= 249)
).astype(int)

print("VIP customers in train:", df_train_final["is_vip"].sum())
print("VIP customers in valid:", df_valid_final["is_vip"].sum())

# ── Define feature matrix ─────────────────────────────────────────────────────
feature_cols = [col for col in df_train_final.columns
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
y_train = df_train_final["revenue_2018_2019"]
X_valid = df_valid_final[feature_cols]
y_valid = df_valid_final["revenue_2018_2019"]

y_train_binary = (y_train > 0).astype(int)
y_valid_binary = (y_valid > 0).astype(int)

print("\nFeature count:", len(feature_cols))
print("X_train shape:", X_train.shape)

VIP customers in train: 4677
VIP customers in valid: 1202

Feature count: 27
X_train shape: (93272, 27)


## 5. Best Approach — LGBM optimizing MAE

In [10]:
# ── LGBM with MAE objective ──────────────────────────────────────────────────
reg_lgbm_mae = LGBMRegressor(
    objective="mae",
    n_estimators=700, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_lgbm_mae.fit(X_train, np.log1p(y_train))

y_pred_lgbm_mae = np.expm1(reg_lgbm_mae.predict(X_valid))
y_pred_lgbm_mae = np.clip(y_pred_lgbm_mae, 0, None)

print(f"MAE (LGBM MAE):      {mean_absolute_error(y_valid, y_pred_lgbm_mae):.2f}")
print(f"Spearman (LGBM MAE): {spearmanr(y_valid, y_pred_lgbm_mae).statistic:.3f}")

MAE (LGBM MAE):      64.02
Spearman (LGBM MAE): 0.385


In [11]:
# ── Grid search over LGBM hyperparameters ────────────────────────────────────
# Best params: learning_rate=0.01, max_depth=6, n_estimators=1000, subsample=0.7
# Grid search (commented out):
# param_grid_lgbm = {"n_estimators": [500,700,1000], "max_depth": [4,6],
#                    "learning_rate": [0.01,0.05], "subsample": [0.7,0.8]}
# grid_search_lgbm = GridSearchCV(LGBMRegressor(objective="mae", colsample_bytree=0.8,
#                                 random_state=42, n_jobs=-1, verbose=-1),
#                                 param_grid_lgbm, cv=5,
#                                 scoring="neg_mean_absolute_error", verbose=1, n_jobs=-1)
# grid_search_lgbm.fit(X_train, np.log1p(y_train))
# → learning_rate=0.01, max_depth=6, n_estimators=1000, subsample=0.7

reg_lgbm_best = LGBMRegressor(
    objective="mae",
    learning_rate=0.01, max_depth=6, n_estimators=1000,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_lgbm_best.fit(X_train, np.log1p(y_train))

y_pred_lgbm_best = np.expm1(reg_lgbm_best.predict(X_valid))
y_pred_lgbm_best = np.clip(y_pred_lgbm_best, 0, None)

print(f"MAE (LGBM MAE grid search):      {mean_absolute_error(y_valid, y_pred_lgbm_best):.2f}")
print(f"Spearman (LGBM MAE grid search): {spearmanr(y_valid, y_pred_lgbm_best).statistic:.3f}")

MAE (LGBM MAE grid search):      63.13
Spearman (LGBM MAE grid search): 0.391


In [12]:
# ── Feature importance ────────────────────────────────────────────────────────
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": reg_lgbm_best.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance.to_string(index=False))

                feature  importance
           recency_days        4603
avg_days_between_orders        2412
        pct_orders_2017        2338
   n_product_categories        2025
          total_revenue        2012
              buys_kids        1939
  items_per_order_gross        1770
               n_brands        1577
            total_items        1474
             avg_ticket        1415
               buys_men        1203
              n_genders        1003
              frequency         928
          discount_rate         923
 total_discounted_items         705
std_days_between_orders         635
             buys_women         604
      is_one_time_buyer         482
            return_rate         461
    items_per_order_net         349
          revenue_trend         321
           pct_web_only         316
          total_returns         241
            buys_in_jul          98
            buys_in_jan          92
                 is_vip          72
            buys_outlet     

In [13]:
# ── Feature selection: keep features with importance >= 200 ──────────────────
# buys_outlet has importance=0, n_genders and is_vip have very low importance
# Removing low-importance features reduces noise
top_features = feature_importance[feature_importance["importance"] >= 200]["feature"].tolist()
print(f"Features with importance >= 200: {len(top_features)}")

X_train_top = df_train_final[top_features]
X_valid_top = df_valid_final[top_features]

Features with importance >= 200: 23


In [14]:
# candidate_params = [
#     {"num_leaves": nl, "min_child_samples": mc, "learning_rate": lr,
#      "n_estimators": ne, "subsample": 0.8, "colsample_bytree": 0.8,
#      "reg_alpha": 0.5, "reg_lambda": 2.0}
#     for nl in [25, 31, 40]
#     for mc in [15, 20, 30]
#     for lr in [0.003, 0.005, 0.008]
#     for ne in [4000, 5000, 6000]
# ]

# print(f"Total combinations: {len(candidate_params)}")  # 81 combinaciones

# results = []
# for i, params in enumerate(candidate_params, 1):
#     model = LGBMRegressor(objective="mae", random_state=42, n_jobs=-1, verbose=-1, **params)
#     model.fit(X_train_top, np.log1p(y_train))
#     pred = np.clip(np.expm1(model.predict(X_valid_top)), 0, None)
#     mae  = mean_absolute_error(y_valid, pred)
#     sp   = spearmanr(y_valid, pred).statistic
#     results.append({"trial": i, "mae": mae, "spearman": sp, **params})
#     if i % 10 == 0:
#         print(f"Trial {i}/81 — best so far: {min(r['mae'] for r in results):.4f}")

# results_df = pd.DataFrame(results).sort_values("mae")
# print("\nTop 5 results:")
# print(results_df.head(5).to_string(index=False))

# # Top 5 results:
# #  trial       mae  spearman  num_leaves  min_child_samples  learning_rate  n_estimators  subsample  colsample_bytree  reg_alpha  reg_lambda
# #     74 63.027273  0.393199          40                 30          0.003          5000        0.8               0.8        0.5         2.0
# #     75 63.036531  0.393664          40                 30          0.003          6000        0.8               0.8        0.5         2.0
# #     51 63.037201  0.391101          31                 30          0.005          6000        0.8               0.8        0.5         2.0
# #     50 63.039194  0.391729          31                 30          0.005          5000        0.8               0.8        0.5         2.0
# #     12 63.041584  0.392087          25                 20          0.003          6000        0.8               0.8        0.5         2.0

In [15]:
# ── BEST MODEL —
reg_final = LGBMRegressor(
    objective="mae",
    num_leaves=40,
    min_child_samples=30,
    learning_rate=0.003,
    n_estimators=5000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
reg_final.fit(X_train_top, np.log1p(y_train))

y_pred_final = np.expm1(reg_final.predict(X_valid_top))
y_pred_final = np.clip(y_pred_final, 0, None)

print(f"MAE (BEST MODEL):      {mean_absolute_error(y_valid, y_pred_final):.2f}")
print(f"Spearman (BEST MODEL): {spearmanr(y_valid, y_pred_final).statistic:.3f}")

MAE (BEST MODEL):      63.03
Spearman (BEST MODEL): 0.393


In [16]:
# ── Prepare raw transactions for test customers ───────────────────────────────
df_test_raw = pd.read_csv("data/transactions_2016_2017.csv", low_memory=False,
                           parse_dates=["order_date", "pack_date"])
df_test_customers = pd.read_csv("data/customer_clv_test.csv")  # ← línea que faltaba

# Create is_returned column (1 if returned, 0 if not)
df_test_raw["is_returned"] = (df_test_raw["returned_to_shop_id"].notna()).astype(int)

# Filter only test customers
test_cust_ids = df_test_customers["cust_id"].astype(str).tolist()
df_test_transactions = df_test_raw[df_test_raw["cust_id"].astype(str).isin(test_cust_ids)].copy()

print(f"Test transactions: {df_test_transactions.shape}")
print(f"Test customers found: {df_test_transactions['cust_id'].nunique()}")
print(f"Test customers expected: {len(test_cust_ids)}")

# Build features
df_test_features = build_features(df_test_transactions)
df_test_final    = df_test_customers.merge(df_test_features, on="cust_id", how="left")

print(f"\nTest final shape: {df_test_final.shape}")
print(f"NaNs en top_features: {df_test_final[top_features].isna().sum().sum()}")

Test transactions: (69046, 27)
Test customers found: 29148
Test customers expected: 29148

Test final shape: (29148, 27)
NaNs en top_features: 68057


In [17]:
# NaN handling — same as train
cols_fill_zero = ["frequency", "total_revenue", "avg_ticket",
                  "items_per_order_net", "is_one_time_buyer"]
df_test_final[cols_fill_zero] = df_test_final[cols_fill_zero].fillna(0)
df_test_final["revenue_trend"] = df_test_final["revenue_trend"].fillna(0)
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    df_test_final[col] = df_test_final[col].fillna(df_train_final[col].median())

# VIP feature — same thresholds as train
df_test_final["is_vip"] = (
    (df_test_final["frequency"] >= 4) &
    (df_test_final["total_revenue"] >= 249)
).astype(int)

# Predict with best model (LGBM MAE Trial 3 — MAE 63.05)
X_test      = df_test_final[top_features]
y_pred_test = np.expm1(reg_final.predict(X_test))
y_pred_test = np.clip(y_pred_test, 0, None)

# Save
submission = pd.DataFrame({
    "cust_id":    df_test_final["cust_id"],
    "prediction": y_pred_test
})
submission.to_csv("submission_group21.csv", index=False)

print(f"Submission saved! {submission.shape[0]} customers")
print(f"Predicciones únicas: {pd.Series(y_pred_test).nunique()}")
print(f"Mean: {y_pred_test.mean():.2f} | Max: {y_pred_test.max():.2f} | Zeros: {(y_pred_test==0).sum()}")
print(submission.head(10))

Submission saved! 29148 customers
Predicciones únicas: 12724
Mean: 24.20 | Max: 809.24 | Zeros: 16397
            cust_id  prediction
0  2dfoualegmpt6x2h  345.490729
1  d2q2stjpnzld7a4r    5.053875
2  cojscuqlpylhclv2    0.000000
3  vntezlhi2ryvxk6m   82.995734
4  jgy4ytjkdr2b75wf   75.228380
5  z5556xsae7hjyzns    0.000000
6  bsgb4hg5tvtm6c34    0.000151
7  w6helbfkvtpppzjx   52.923915
8  kklab3poqqxo5c2w    0.000000
9  unhzujie5tubmofh    0.088181
